In [12]:
import geopandas as gpd
import create_dataset
import numpy as np
from shapely.geometry import Polygon
import xarray as xr
import zarr
import os
from pathlib import Path

# some_file.py
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, '../../../scripts/')

import directories as directories
import metadata as metadata
import load_data as load_data
import plotting as plotting

## -- inputs.


resolution = 100
domain = 'cues'


## load metadata
# load config
## --------------- CUES
cues_cfg = metadata.load_yaml(Path(f"../../../../configs/CUES.yaml"))
# load point geometry
cues_lat,cues_lon,cues_x,cues_y,cues_crs = metadata.get_point_geography(cues_cfg,'cues')
sesame_lat,sesame_lon,sesame_x,sesame_y,sesame_crs = metadata.get_point_geography(cues_cfg,'sesame')
mhp_lat,mhp_lon,mhp_x,mhp_y,mhp_crs = metadata.get_point_geography(cues_cfg,'mhp')
# load snowmodel filepath
sm_cues_biased_fpath = metadata.get_snowmodel_fpath(cues_cfg,'biased')
sm_cues_corrected_fpath = metadata.get_snowmodel_fpath(cues_cfg,'corrected')
# load snowmodel filepath
sentinel_cues_10m_fpath = metadata.get_sentinel_fpath(cues_cfg,'10m')
sentinel_cues_40m_fpath = metadata.get_sentinel_fpath(cues_cfg,'40m')
# load snowmodel filepath
sm_cues_OBS_swe_fpath = metadata.get_cuesOBS_fpath(cues_cfg,'swe')
sm_cues_OBS_sweMelt_fpath = metadata.get_cuesOBS_fpath(cues_cfg,'swe_melt')

## --------------- KSP
ksp_cfg = metadata.load_yaml(Path(f"../../../../configs/KSP.yaml"))
ksp_lat,ksp_lon,ksp_x,ksp_y,ksp_crs = metadata.get_point_geography(ksp_cfg,'ksp')

## --------------- KSP
tuolumne_cfg = metadata.load_yaml(Path(f"../../../../configs/Tuolumne.yaml"))
tuolumneGauge_lat,tuolumneGauge_lon,tuolumneGauge_x,tuolumneGauge_y,tuolumneGauge_crs = metadata.get_point_geography(tuolumne_cfg,'tuolumne_gauge')
lyellGauge_lat,lyellGauge_lon,lyellGauge_x,lyellGauge_y,lyellGauge_crs = metadata.get_point_geography(tuolumne_cfg,'lyell_gauge')
danaGauge_lat,danaGauge_lon,danaGauge_x,danaGauge_y,danaGauge_crs = metadata.get_point_geography(tuolumne_cfg,'dana_gauge')
tuolumnePillow_lat,tuolumnePillow_lon,tuolumnePillow_x,tuolumnePillow_y,tuolumnePillow_crs = metadata.get_point_geography(tuolumne_cfg,'tuolumne_pillow')
danaPillow_lat,danaPillow_lon,danaPillow_x,danaPillow_y,danaPillow_crs = metadata.get_point_geography(tuolumne_cfg,'dana_pillow')

# cues
mammoth_geog_fpath,aso_crs = metadata.get_shapefile_fpath(cues_cfg,"shapefile_fpath","zoom_1")
cues_geog_fpath,aso_crs = metadata.get_shapefile_fpath(cues_cfg,"shapefile_fpath","zoom_2")
# ksp
ksp_zoom1_geog_fpath,aso_crs = metadata.get_shapefile_fpath(ksp_cfg,"shapefile_fpath","zoom_1")
ksp_zoom2_geog_fpath,aso_crs = metadata.get_shapefile_fpath(ksp_cfg,"shapefile_fpath","zoom_2")
# tuolumne
tuolumne_geog_fpath,aso_crs = metadata.get_shapefile_fpath(tuolumne_cfg,"shapefile_fpath","zoom_1")
dana_watershed_fpath = metadata.get_watershed_fpath(tuolumne_cfg,'dana_watershed')
lyell_watershed_fpath = metadata.get_watershed_fpath(tuolumne_cfg,'lyell_watershed')
tuolumne_watershed_fpath = metadata.get_watershed_fpath(tuolumne_cfg,'tuolumne_watershed')
# ASO
aso_cfg = metadata.load_yaml(Path(f"../../../../configs/ASO.yaml"))
uscatm_geog_fpath,aso_crs = metadata.get_shapefile_fpath(aso_cfg,"basin_shapefile","USCATM")
uscasj_geog_fpath,aso_crs = metadata.get_shapefile_fpath(aso_cfg,"basin_shapefile","USCASJ") 
# read shapefile
## mammoth
mammoth_geog_gdf = gpd.read_file(mammoth_geog_fpath)
mammoth_proj_gdf = mammoth_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## cues
cues_geog_gdf = gpd.read_file(cues_geog_fpath)
cues_proj_gdf = cues_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## mammoth
ksp_zoom1_geog_gdf = gpd.read_file(ksp_zoom1_geog_fpath)
ksp_zoom1_proj_gdf = ksp_zoom1_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## cues
ksp_zoom2_geog_gdf = gpd.read_file(ksp_zoom2_geog_fpath)
ksp_zoom2_proj_gdf = ksp_zoom2_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## uscatm
uscatm_geog_gdf = gpd.read_file(uscatm_geog_fpath)
uscatm_proj_gdf = uscatm_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## uscatm
uscasj_geog_gdf = gpd.read_file(uscasj_geog_fpath)
uscasj_proj_gdf = uscasj_geog_gdf.to_crs(f'EPSG:{aso_crs}')
## tuolumne
tuolumne_geog_gdf = gpd.read_file(tuolumne_geog_fpath)
tuolumne_proj_gdf = tuolumne_geog_gdf.to_crs(f'EPSG:{aso_crs}')


if domain == 'cues':
    bbox_gdf = mammoth_geog_gdf
    bbox_gdf.attrs = {'filename':mammoth_geog_fpath}
    geojson = 'mammoth.json'
elif domain == 'tuolumne':
    bbox_gdf = tuolumne_geog_gdf
    bbox_gdf.attrs = {'filename':tuolumne_geog_fpath}
    geojson = 'tuolumne.json'
else:
    print('Passed incorrect domain! Please resubmit!')
    sys.exit()


reference_scenes_aggregation_technique = 'median' # mean / median / max


In [13]:
start_date = '2015-10-01'
end_date = '2016-10-01'
year = int(end_date[0:4])
ref_start = f'{year}-07-15'
ref_end = f'{year}-09-01'

In [14]:
save_dir = f'/glade/derecho/scratch/rossamower/snow/snowmodel/school/uw/cues/data/sentinel/{domain}/{resolution}_m/{year}/'
reference_dir = f'{save_dir}melt_thresh/reference/'
if not os.path.exists(save_dir): os.makedirs(save_dir)
if not os.path.exists(reference_dir): os.makedirs(reference_dir)

time_reference_scenes = slice(ref_start,ref_end) 
print(f'domain {domain}; processing {year}; ref {ref_start}- {ref_end}; resolution {resolution}')

domain cues; processing 2016; ref 2016-07-15- 2016-09-01; resolution 100


In [ ]:
cues_sm_corrected_rpre_da = load_data.load_multidirectory_snowmodel_output(sm_cues_corrected_fpath,var = 'rpre')
cues_sm_corrected_swed_da = load_data.load_multidirectory_snowmodel_output(sm_cues_corrected_fpath,var = 'swed')

In [ ]:
sm_rpre_ref_da = cues_sm_corrected_rpre_da.where(cues_sm_corrected_rpre_da.time >= np.datetime64(ref_start),drop = True).where(cues_sm_corrected_rpre_da.time <= np.datetime64(ref_end),drop = True)
sm_swed_ref_da = cues_sm_corrected_swed_da.where(cues_sm_corrected_swed_da.time >= np.datetime64(ref_start),drop = True).where(cues_sm_corrected_swed_da.time <= np.datetime64(ref_end),drop = True)

In [ ]:
import matplotlib.pyplot as plt
fig,ax = plt.subplots()
sm_rpre_ref_da.plot(ax=ax,color = 'C0')
ax2 = ax.twinx()
sm_swed_ref_da.plot(ax=ax2,color = 'C1')

In [15]:
weekly_dates = np.arange(f'{year-1}-10-01', f'{year}-10-01', 7, dtype='datetime64[D]')
for date in weekly_dates:
    time_target_scene = str(date)

    composite_reference_scene,ratio_images = create_dataset.get_composite_images(bbox_gdf, year, time_reference_scenes, time_target_scene, reference_scenes_aggregation_technique, resolution)
    dataset = composite_reference_scene.to_dataset(name='reference_bscatter')
    dataset = dataset.rio.write_crs(bbox_gdf.estimate_utm_crs().to_epsg())
    dataset.load()
    # bscatter_ref = 10*np.log10(dataset.rio.clip(cues_proj_gdf.geometry).isel({'y':0,'x':0,'band':0,'sat:relative_orbit':2})['reference_bscatter'].values) # 144; vh
    # bscatter_ref = 10*np.log10(dataset.rio.clip(tuolumne_proj_gdf.geometry).isel({'band':0,'sat:relative_orbit':2})['reference_bscatter'].values) # 144; vh
    # print(time_target_scene,bscatter_ref)
    break



In [ ]:
dataset

In [ ]:
for year in range(2017,2021):
    weekly_dates = np.arange(f'{year-1}-10-01', f'{year}-10-01', 7, dtype='datetime64[D]')
    for date in weekly_dates:
        time_target_scene = str(date)

        composite_reference_scene,ratio_images = create_dataset.get_composite_images(bbox_gdf, year, time_reference_scenes, time_target_scene, reference_scenes_aggregation_technique, resolution)
        dataset = composite_reference_scene.to_dataset(name='reference_bscatter')
        dataset = dataset.rio.write_crs(bbox_gdf.estimate_utm_crs().to_epsg())
        bscatter_ref = 10*np.log10(dataset.rio.clip(cues_proj_gdf.geometry).isel({'y':0,'x':0,'band':0,'sat:relative_orbit':2})['reference_bscatter'].values) # 144; vh
        print(time_target_scene,bscatter_ref)
        break

In [ ]:
dataset

In [16]:
dataset.to_netcdf(f'{reference_dir}reference_images_{ref_start}_{ref_end}.nc')

In [7]:
f'{reference_dir}reference_images_{ref_start}_{ref_end}.nc'

'/glade/derecho/scratch/rossamower/snow/snowmodel/school/uw/cues/data/sentinel/tuolumne/100_m/2017/melt_thresh/reference/reference_images_2017-07-15_2017-09-01.nc'

In [ ]:
f'{reference_dir}reference_images_{ref_start}_{ref_end}.nc'

In [ ]:
!ls $save_dir/melt_thresh/

In [ ]:
ratio_images = create_dataset.get_ratio_images(bbox_gdf, year, time_reference_scenes, time_target_scene, reference_scenes_aggregation_technique, resolution) # 10m but we pull at 100m
dataset = ratio_images.to_dataset(name='ratio_images')
dataset = dataset.rio.write_crs(bbox_gdf.estimate_utm_crs().to_epsg())